# 開放封閉原則 (Open/Closed Principle, OCP)
## 定義
軟體實體（如類別、模組、函數等）應該對擴展開放，對修改封閉。

## 判斷導入時機

- Do：
當需要為系統添加新功能，而不希望影響現有代碼的穩定性時。
希望通過`擴展`而非修改來適應新需求時。

- Don't：
當過度設計導致不必要的抽象，增加系統複雜性時。


既有類別已經定義清楚，處於一個強調穩定的狀態
或是你需要擴充現有類別，加入新需求的屬性或方法
又或是擔心修改現有程式碼會破壞現有系統，於是擴充它而不改原本的code

不過，並不是所有類別都需要可擴充性，這要看需求、看你對系統本身的理解。當然，若你的程式碼一開始就不具備可擴充性，其實也沒關係，我們可以透過重構的技術讓程式碼變得可擴充

OCP有很多實作方式，可以透過繼承，保留原本的code，並撰寫新的code，套用OCP有其成本也有其價值在，價值就是你不會改到原本的code，但成本就是-類別越來越多，若能從類別名稱判斷差異(v1、v2XDD)，可讀性不一定會差。

----


## 完整案例

### 需求情境

設計訂單系統需要支持不同的折扣策略

1. 百分比折扣（PercentageDiscount）：按照總金額的百分比計算折扣。
2. 固定金額折扣（FixedDiscount）：直接減去固定金額。
3. 可以在未來輕鬆增加新的折扣類型，例如滿減折扣（ThresholdDiscount）。

### 初始設計
問題：每次新增折扣類型，都需要修改 Order 類別，這會影響系統的穩定性和可測試性。

In [5]:
class Item:
    def __init__(self, name, price):
        self.name = name
        self.price = price

class Order:
    def __init__(self, items, discount_type, discount_value):
        self.items = items
        self.discount_type = discount_type
        self.discount_value = discount_value

    def calculate_total(self):
        total = sum(item.price for item in self.items)
        if self.discount_type == 'percentage':
            total -= total * self.discount_value / 100
        elif self.discount_type == 'fixed':
            total -= self.discount_value
        return total

# 模擬使用
items = [
    Item(name="Laptop", price=1500),
    Item(name="Mouse", price=50),
    Item(name="Keyboard", price=100)
]

order = Order(items, 'percentage', 10)  # 百分比折扣
print(f"Order Total with Discount: {order.calculate_total()}")

order = Order(items, 'fixed', 100)  # 固定金額折扣
print(f"Order Total with Discount: {order.calculate_total()}")




Order Total with Discount: 1485.0
Order Total with Discount: 1550


> 問題分析

違反 OCP：新增折扣類型時，必須修改 Order 類別中的 calculate_total 方法。
代碼可讀性差：折扣邏輯直接寫在 Order 類別中，降低了代碼的可測試性和維護性。

### 重構設計
解決方案：通過引入策略模式 (Strategy Pattern)，將折扣邏輯抽象為一個接口 DiscountStrategy，並為每種折扣實現具體的子類別。

In [6]:
from abc import ABC, abstractmethod

# 抽象折扣策略
class DiscountStrategy(ABC):
    @abstractmethod
    def apply_discount(self, total):
        pass

# 百分比折扣策略
class PercentageDiscount(DiscountStrategy):
    def __init__(self, percentage):
        self.percentage = percentage

    def apply_discount(self, total):
        return total - (total * self.percentage / 100)

# 固定金額折扣策略
class FixedDiscount(DiscountStrategy):
    def __init__(self, amount):
        self.amount = amount

    def apply_discount(self, total):
        return total - self.amount

# 訂單類別
class Order:
    def __init__(self, items, discount_strategy: DiscountStrategy):
        self.items = items
        self.discount_strategy = discount_strategy

    def calculate_total(self):
        total = sum(item.price for item in self.items)
        return self.discount_strategy.apply_discount(total)

# 模擬使用
items = [
    Item(name="Laptop", price=1500),
    Item(name="Mouse", price=50),
    Item(name="Keyboard", price=100)
]

# 使用百分比折扣
percentage_discount = PercentageDiscount(10)  # 10% 折扣
order = Order(items, percentage_discount)
print(f"Order Total with Percentage Discount: {order.calculate_total()}")

# 使用固定金額折扣
fixed_discount = FixedDiscount(100)  # 固定折扣 $100
order = Order(items, fixed_discount)
print(f"Order Total with Fixed Discount: {order.calculate_total()}")


Order Total with Percentage Discount: 1485.0
Order Total with Fixed Discount: 1550


### 擴展設計
假設需要新增一種 滿減折扣策略：訂單金額超過某個閾值時，減去固定金額。此時只需要新增一個類別，而無需修改現有代碼。

In [7]:
# 滿減折扣策略
class ThresholdDiscount(DiscountStrategy):
    def __init__(self, threshold, discount_amount):
        self.threshold = threshold
        self.discount_amount = discount_amount

    def apply_discount(self, total):
        if total > self.threshold:
            return total - self.discount_amount
        return total

# 使用滿減折扣
threshold_discount = ThresholdDiscount(threshold=1000, discount_amount=200)
order = Order(items, threshold_discount)
print(f"Order Total with Threshold Discount: {order.calculate_total()}")


Order Total with Threshold Discount: 1450


### 重構優點
- 高擴展性：新增折扣類型時，只需新增對應的策略類別，不影響現有代碼。
- 高維護性：每個折扣邏輯封裝在獨立的類別中，單一職責清晰。
- 高可測試性：可以單獨測試每種折扣策略，減少測試依賴。
- 滿足 OCP：系統對擴展開放（新增折扣策略），對修改封閉（Order 類別不需要修改）。


適用場景
- 系統需要頻繁新增功能（如新的折扣類型）。
- 希望確保現有代碼穩定，降低因修改引入的風險。